# Continuous evaluation setup (SDK path)

Creates an eval object with three built-in evaluators and a rule that runs on every response completion for the demo agent.

**Important:** the continuous-eval rule's async processor is unreliable in the current preview (runs may not materialize even when responses are stored with `store=True`). The reliable way to populate the Foundry portal **Evaluations** pane is the one-shot batch eval over traces, run by `scripts/20-agent-batch-eval.py` (or the last cell below). That uses the `azure_ai_traces_preview` data source against App Insights and completes in 1-3 minutes.

Source: https://learn.microsoft.com/azure/foundry/observability/how-to/how-to-monitor-agents-dashboard

In [ ]:
import json as _json
import os, pathlib
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Discover the active azd env dynamically: prefer AZURE_ENV_NAME, else read
# the 'defaultEnvironment' from agent/.azure/config.json, else fall back to
# the first existing env directory.
AGENT_DIR = pathlib.Path('..') / 'agent'
_azd_base = AGENT_DIR / '.azure'
_env_name = os.environ.get('AZURE_ENV_NAME', '')
if not _env_name:
    _config = _azd_base / 'config.json'
    if _config.exists():
        try:
            _env_name = _json.loads(_config.read_text()).get('defaultEnvironment', '')
        except Exception:
            _env_name = ''
if _env_name and (_azd_base / _env_name / '.env').exists():
    AZD_ENV = _azd_base / _env_name / '.env'
else:
    AZD_ENV = next(
        (p / '.env' for p in sorted(_azd_base.iterdir()) if p.is_dir() and (p / '.env').exists()),
        _azd_base / 'default' / '.env',
    )
if AZD_ENV.exists():
    load_dotenv(AZD_ENV)
load_dotenv()

endpoint = os.environ['AZURE_AI_PROJECT_ENDPOINT']
agent_name = os.environ.get('AZURE_AI_AGENT_NAME', 'agent-framework-agent-basic-responses')
model_deployment = os.environ.get('AZURE_AI_MODEL_DEPLOYMENT_NAME') or os.environ['MODEL_DEPLOYMENT_NAME']
print('azd env:        ', _env_name or '(fallback)')
print('endpoint:       ', endpoint)
print('agent_name:     ', agent_name)
print('model_deployment:', model_deployment)

endpoint: https://ai-account-eyfs7o7lvdq7e.services.ai.azure.com/api/projects/ai-project-aiobs-foundry-20260520
agent_name: agent-framework-agent-basic-responses
model_deployment: gpt-4o-mini


In [ ]:
credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project.get_openai_client()

data_source_config = {'type': 'azure_ai_source', 'scenario': 'responses'}
# Source fields must match what the azure_ai_traces_preview data source emits
# per trace item: query, response, tool_calls, tool_definitions. Using
# {{item.input}} or {{item.output}} here silently produces empty rows and
# every evaluator errors with "Missing inputs for line N: 'data.input, data.output'".
common_qr = {'query': '{{item.query}}', 'response': '{{item.response}}'}
testing_criteria = [
    {'type': 'azure_ai_evaluator', 'name': 'intent_resolution',
     'evaluator_name': 'builtin.intent_resolution',
     'data_mapping': common_qr,
     'initialization_parameters': {'deployment_name': model_deployment}},
    {'type': 'azure_ai_evaluator', 'name': 'tool_call_accuracy',
     'evaluator_name': 'builtin.tool_call_accuracy',
     'data_mapping': {
         'query': '{{item.query}}',
         'response': '{{item.response}}',
         'tool_definitions': '{{item.tool_definitions}}',
         'tool_calls': '{{item.tool_calls}}',
     },
     'initialization_parameters': {'deployment_name': model_deployment}},
    {'type': 'azure_ai_evaluator', 'name': 'violence',
     'evaluator_name': 'builtin.violence',
     'data_mapping': common_qr},
]
eval_object = openai_client.evals.create(
    name='Demo continuous eval',
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
print('eval id:', eval_object.id)
EVAL_ID = eval_object.id

eval id: eval_e2c07420659440d9b846de1affb3a434


In [3]:
from azure.ai.projects.models import (
    EvaluationRule,
    ContinuousEvaluationRuleAction,
    EvaluationRuleFilter,
    EvaluationRuleEventType,
)
rule = project.evaluation_rules.create_or_update(
    id='demo-continuous-eval',
    evaluation_rule=EvaluationRule(
        display_name='Demo continuous eval',
        description='Runs on every response completion for the demo agent',
        action=ContinuousEvaluationRuleAction(eval_id=EVAL_ID, max_hourly_runs=100),
        event_type=EvaluationRuleEventType.RESPONSE_COMPLETED,
        filter=EvaluationRuleFilter(agent_name=agent_name),
        enabled=True,
    ),
)
print('rule id:', rule.id)

rule id: demo-continuous-eval


In [4]:
import json, pathlib
pathlib.Path('../artifacts').mkdir(exist_ok=True)
pathlib.Path('../artifacts/continuous-eval.json').write_text(
    json.dumps({'eval_id': EVAL_ID, 'rule_id': rule.id, 'agent_name': agent_name}, indent=2)
)
print('Saved artifacts/continuous-eval.json')

Saved artifacts/continuous-eval.json


In [ ]:
# Verify by listing recent runs (the rule's async processor is unreliable in
# preview; if this stays empty, run the batch eval cell below instead).
runs = openai_client.evals.runs.list(eval_id=EVAL_ID, order='desc', limit=10)
for r in runs.data:
    print(r.id, r.status)

## Populate the Evaluations pane immediately (batch eval over traces)

Runs the same eval definition created above against the last 2h of agent traces in App Insights using the `azure_ai_traces_preview` data source. This is the reliable path to surface results in the Foundry portal **Evaluations** pane.

Requires traces to already be flowing (i.e. the agent has been invoked recently). Completes in 1-3 minutes.

In [ ]:
import time

LOOKBACK_HOURS = 2
MAX_TRACES = 20

eval_run = openai_client.evals.runs.create(
    eval_id=EVAL_ID,
    name=f'demo-traces-batch-{int(time.time())}',
    data_source={
        'type': 'azure_ai_traces_preview',
        'agent_name': agent_name,
        'lookback_hours': LOOKBACK_HOURS,
        'max_traces': MAX_TRACES,
    },
)
print('run id:', eval_run.id, 'status:', eval_run.status)
RUN_ID = eval_run.id

# Poll to completion so the pane shows results before this cell returns.
deadline = time.time() + 8 * 60
last_status = eval_run.status
while time.time() < deadline:
    eval_run = openai_client.evals.runs.retrieve(run_id=RUN_ID, eval_id=EVAL_ID)
    if eval_run.status != last_status:
        print(' status:', eval_run.status)
        last_status = eval_run.status
    if eval_run.status in ('completed', 'failed', 'canceled'):
        break
    time.sleep(15)

rc = eval_run.result_counts
print(f'final: status={eval_run.status} total={rc.total} passed={rc.passed} failed={rc.failed} errored={rc.errored}')
if getattr(eval_run, 'report_url', None):
    print('report:', eval_run.report_url)